<div id="singlestore-header" style="display: flex; background-color: rgba(235, 249, 245, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/bolt.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Tune a Real-Time Ad Platform with Resource Pools, RETURNING and Projections</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook can be run on a Free Starter Workspace. To create a Free Starter Workspace navigate to <tt>+</tt> using the left nav. You can also use your existing Standard or Premium workspace with this Notebook.</p>
    </div>
</div>

## The story

It's the Monday after Super Bowl night at **AdVenture Inc.**, the New York real-time ad startup where you are still the only developer. Search never blinked during the game, and Morgan Blaze, the CMO, told the board you're a genius.

The traffic never went home, though. One database now **serves ads** in milliseconds, **charges advertisers' budgets** as impressions land, and **runs campaign reports**, plus an AI agent that writes its own SQL, all against the same tables.

By 8 AM there are five tickets in your queue. SingleStore's August 2026 release (engine 10.4) has a feature for each one. This notebook builds the platform on real data and measures every fix, before and after:

| Ticket | Problem | Feature | What you will measure |
|---|---|---|---|
| ADV-311, Finance | Charging a budget and reading the new balance takes two round trips, and the read can see another server's charge | `UPDATE ... RETURNING` | Round trips, and how many concurrent writers see the wrong balance |
| ADV-312, Privacy | Expiring audience segments means a `SELECT` then a `DELETE`, with a gap between them | `DELETE ... RETURNING` | One statement that deletes and reports what it removed |
| ADV-313, Sales | One advertiser's dashboard scans every row, because the table is laid out by time and subscriber | Projections | Rows scanned with and without a projection, and what it costs on writes |
| ADV-314, Data Engineering | Setup scripts break on re-run once a projection exists | `DROP PROJECTION IF EXISTS` | A teardown that is safe to run twice |
| ADV-315, Morgan | A heavy report, and now an AI agent, slows down ad serving | CPU resource pools | Ad-serving latency with reports in the default pool, then in their own pool |

**Lab notes** boxes record what we measured across repeated runs. Absolute times depend on your workspace, so they are given as ratios; run the cells more than once.

## What's in this notebook:

1. Load the ad platform's data.
2. Charge budgets with `UPDATE ... RETURNING`.
3. Expire audience segments with `DELETE ... RETURNING`.
4. Give each advertiser's dashboard its own layout with a projection.
5. Make projection DDL re-runnable with `DROP PROJECTION IF EXISTS`.
6. Keep reports and AI agents from slowing ad serving with resource pools.
7. Clean up.
8. Wrap up: what each feature is and when to use it.

Sections 1-5 run on any workspace. Section 6 needs a Standard or Premium Workspace, because resource pools need privileges a Free Starter Workspace does not grant; the notebook skips it there.

## Questions?

Reach out to us through our [forum](https://www.singlestore.com/forum).

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p> If you have a Free Starter Workspace deployed already, select the database from drop-down menu at the top of this notebook. It updates the <tt>connection_url</tt> to connect to that database.</p>
    </div>
</div>

Create a database for the examples below. The cell keeps an existing `ad_platform` instead of dropping it, because dropping the database a notebook session is using leaves that session with no database. Section 1 clears out its tables anyway.

In [1]:
shared_tier_check = %sql show variables like 'is_shared_tier'
if not shared_tier_check or shared_tier_check[0][1] == 'OFF':
    %sql CREATE DATABASE IF NOT EXISTS ad_platform;
    %sql USE ad_platform;

current_database = %sql SELECT DATABASE()
DATABASE = current_database[0][0]
if not DATABASE:
    raise RuntimeError('No database selected. On a Free Starter Workspace, pick your database in the '
                       'drop-down menu at the top of this notebook, then run this cell again.')
print(f'Using database {DATABASE}')

On a Standard or Premium Workspace there is nothing to pick in the drop-down: the notebook switches to `ad_platform` itself. On a Free Starter Workspace it uses the database you selected.

## 1. Load the ad platform's data.

Nobody tests fixes on production the morning after the Super Bowl, so start with a copy of AdVenture's platform.

The data is SingleStore's public [real-time digital marketing demo](https://github.com/singlestore-labs/demo-realtime-digital-marketing): 3 million ad requests and 500 thousand purchases, streamed from S3 with pipelines into five tables:

- `requests` and `purchases` -- the raw event streams, sharded by subscriber.
- `offers` -- one campaign per advertiser, with a maximum bid, a budget and spend so far.
- `notifications` -- every ad shown: when, to whom, which offer, and what it cost. Dashboards and reports read this table.
- `subscriber_segments` -- audience memberships such as `buyer:Photobug`, each with an expiry time.

The parquet files have no timestamps, and the demo's offer and segment files are no longer published, so the load derives timestamps, 356 offers (one per vendor) and 3.5 million segment memberships from the streams. It uses `CRC32`, not `RAND()`, so every run produces the same data.

In [2]:
import json
import random
import statistics
import threading
import time

import singlestoredb as s2


def connect():
    """Open a connection to the notebook's database.

    New connections start in whatever database the drop-down menu selects,
    which may be none, so select ``DATABASE`` explicitly.

    Returns
    -------
    Connection
        An open connection with ``DATABASE`` selected.

    """
    conn = s2.connect()
    with conn.cursor() as cur:
        cur.execute(f'USE {DATABASE}')
    return conn


def query(sql, args=None):
    """Run one statement on a fresh connection and return its rows.

    Parameters
    ----------
    sql : str
        The statement to run.
    args : tuple, optional
        Parameters for the statement.

    Returns
    -------
    list of tuple
        The result rows, or an empty list for statements without a result set.

    """
    with connect() as conn, conn.cursor() as cur:
        cur.execute(sql, args)
        return list(cur.fetchall()) if cur.description else []

The notebook is safe to run more than once. Section 4 adds a projection, which blocks `DROP TABLE`, so the next cell removes it first (section 5 explains).

In [3]:
if query("SELECT 1 FROM information_schema.tables WHERE table_schema = DATABASE() AND table_name = 'notifications'"):
    query('DROP PROJECTION IF EXISTS notifications_by_offer ON notifications')

In [4]:
%%sql
DROP PIPELINE IF EXISTS requests_pipeline;
DROP PIPELINE IF EXISTS purchases_pipeline;
DROP TABLE IF EXISTS requests;
DROP TABLE IF EXISTS purchases;
DROP TABLE IF EXISTS offers;
DROP TABLE IF EXISTS notifications;
DROP TABLE IF EXISTS subscriber_segments;

CREATE TABLE requests (
    subscriber_id BIGINT NOT NULL,
    domain TEXT NOT NULL,
    ts DATETIME(6) NOT NULL,
    SORT KEY (ts),
    SHARD KEY (subscriber_id)
);

CREATE TABLE purchases (
    subscriber_id BIGINT NOT NULL,
    vendor TEXT NOT NULL,
    ts DATETIME(6) NOT NULL,
    SORT KEY (ts),
    SHARD KEY (subscriber_id)
);

CREATE TABLE offers (
    offer_id BIGINT NOT NULL,
    customer TEXT NOT NULL,
    enabled BOOLEAN NOT NULL DEFAULT TRUE,
    maximum_bid_cents BIGINT NOT NULL,
    budget_cents BIGINT NOT NULL,
    spend_cents BIGINT NOT NULL DEFAULT 0,
    PRIMARY KEY (offer_id),
    SHARD KEY (offer_id),
    SORT KEY (offer_id)
);

CREATE TABLE notifications (
    ts DATETIME(6) NOT NULL,
    subscriber_id BIGINT NOT NULL,
    offer_id BIGINT NOT NULL,
    cost_cents BIGINT NOT NULL,
    SORT KEY (ts),
    SHARD KEY (subscriber_id)
);

CREATE TABLE subscriber_segments (
    subscriber_id BIGINT NOT NULL,
    segment TEXT NOT NULL,
    expires_at DATETIME(6) NOT NULL,
    SORT KEY (expires_at),
    SHARD KEY (subscriber_id)
);

Each event gets a timestamp in the last 30 days. A pipeline's `SET` clause can only compute from `@variables`, not target columns, so both fields are read into variables first. The load takes a few minutes.

In [5]:
%%sql
CREATE PIPELINE requests_pipeline AS
    LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1m-16p/requests.*'
    CONFIG '{\"region\": \"us-east-1\"}'
    CREDENTIALS '{}'
    INTO TABLE requests FORMAT PARQUET (@sub <- subscriberid, @domain <- domain)
    SET subscriber_id = @sub, domain = @domain,
        ts = NOW(6) - INTERVAL (CRC32(CONCAT(@sub, @domain)) % 2592000) SECOND;

CREATE PIPELINE purchases_pipeline AS
    LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1m-16p/purchases.*'
    CONFIG '{\"region\": \"us-east-1\"}'
    CREDENTIALS '{}'
    INTO TABLE purchases FORMAT PARQUET (@sub <- subscriberid, @vendor <- vendor)
    SET subscriber_id = @sub, vendor = @vendor,
        ts = NOW(6) - INTERVAL (CRC32(CONCAT(@sub, @vendor)) % 2592000) SECOND;

START PIPELINE requests_pipeline FOREGROUND;
START PIPELINE purchases_pipeline FOREGROUND;

Now derive the rest. Every ad request becomes a shown ad, charged at its offer's maximum bid. Subscribers stay in `visits:<domain>` for 28 days after their last visit and in `buyer:<vendor>` for 29 days after their last purchase, so a few percent of memberships have already lapsed; section 3 expires them.

In [6]:
%%sql
INSERT INTO offers (offer_id, customer, maximum_bid_cents, budget_cents)
SELECT ROW_NUMBER() OVER (ORDER BY vendor), vendor,
       5 + CRC32(vendor) % 46,
       100 * (500 + CRC32(REVERSE(vendor)) % 4500)
FROM (SELECT DISTINCT vendor FROM purchases) v;

INSERT INTO notifications (ts, subscriber_id, offer_id, cost_cents)
SELECT r.ts, r.subscriber_id, o.offer_id, o.maximum_bid_cents
FROM (SELECT ts, subscriber_id, 1 + CRC32(CONCAT(subscriber_id, domain)) % 356 AS offer_id FROM requests) r
JOIN offers o ON o.offer_id = r.offer_id;

INSERT INTO subscriber_segments (subscriber_id, segment, expires_at)
SELECT subscriber_id, CONCAT('visits:', domain), MAX(ts) + INTERVAL 28 DAY FROM requests GROUP BY 1, 2
UNION ALL
SELECT subscriber_id, CONCAT('buyer:', vendor), MAX(ts) + INTERVAL 29 DAY FROM purchases GROUP BY 1, 2;

UPDATE offers o
JOIN (SELECT offer_id, SUM(cost_cents) AS s FROM notifications GROUP BY 1) n ON o.offer_id = n.offer_id
SET o.spend_cents = LEAST(n.s, o.budget_cents);

Loading in small batches leaves many small segments, and the background merger that combines them competes for CPU while it runs. Merge them now so the measurements below are steady:

In [7]:
%%sql
OPTIMIZE TABLE requests FULL;
OPTIMIZE TABLE purchases FULL;
OPTIMIZE TABLE notifications FULL;
OPTIMIZE TABLE subscriber_segments FULL;

In [8]:
%%sql
SELECT 'requests' AS table_name, COUNT(*) AS row_count FROM requests
UNION ALL SELECT 'purchases', COUNT(*) FROM purchases
UNION ALL SELECT 'offers', COUNT(*) FROM offers
UNION ALL SELECT 'notifications', COUNT(*) FROM notifications
UNION ALL SELECT 'subscriber_segments', COUNT(*) FROM subscriber_segments;

## 2. Charge budgets with `UPDATE ... RETURNING`.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-311</span> &nbsp;<span style="opacity:0.7">opened 7:52 AM</span><br/>
<b>Priya Raman, Finance:</b> Three advertisers say we charged them past their budget in the fourth quarter last night. Every ad server checks the balance before it serves an impression, so how did we overspend? I owe them an answer by noon, and "the database did it" is not an answer.
</div>

Each ad shown is charged to its offer, and the ad server checks the balance before the next impression. Without `RETURNING` that is two statements: an `UPDATE`, then a `SELECT` for the new value. `RETURNING` hands back the changed rows, with their **new** values, in the same round trip.

In [9]:
%%sql
UPDATE offers SET spend_cents = spend_cents + 17
WHERE offer_id = 42
RETURNING offer_id, customer, spend_cents, budget_cents;

**Latency.** One round trip instead of two. The next cell times both patterns on the same connection.

In [10]:
def median_ms(fn, n=40):
    """Return the median wall-clock time of ``fn()`` in milliseconds.

    Parameters
    ----------
    fn : callable
        Function to time.
    n : int, optional
        Number of runs.

    Returns
    -------
    float
        Median time in milliseconds.

    """
    xs = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        xs.append((time.perf_counter() - t0) * 1000)
    return statistics.median(xs)


with connect() as conn, conn.cursor() as cur:
    def two_statements():
        cur.execute('UPDATE offers SET spend_cents = spend_cents + 1 WHERE offer_id = 42')
        cur.execute('SELECT spend_cents, budget_cents FROM offers WHERE offer_id = 42')
        cur.fetchall()

    def returning():
        cur.execute('UPDATE offers SET spend_cents = spend_cents + 1 WHERE offer_id = 42 '
                    'RETURNING spend_cents, budget_cents')
        cur.fetchall()

    print(f'UPDATE, then SELECT:  {median_ms(two_statements):5.1f} ms')
    print(f'UPDATE ... RETURNING: {median_ms(returning):5.1f} ms')

**Correctness.** Between your `UPDATE` and your `SELECT`, another ad server can charge the same offer, and your `SELECT` reports *its* balance. The next cell runs 8 ad servers at once, each charging offer 42 one cent 50 times and recording the balance it saw. 400 charges should mean 400 different balances.

In [11]:
def charge_offer_42(use_returning, seen):
    """Charge offer 42 one cent, 50 times, recording the balance seen after each charge."""
    with connect() as conn, conn.cursor() as cur:
        for _ in range(50):
            if use_returning:
                cur.execute('UPDATE offers SET spend_cents = spend_cents + 1 WHERE offer_id = 42 RETURNING spend_cents')
            else:
                cur.execute('UPDATE offers SET spend_cents = spend_cents + 1 WHERE offer_id = 42')
                cur.execute('SELECT spend_cents FROM offers WHERE offer_id = 42')
            seen.append(cur.fetchone()[0])


for use_returning in (False, True):
    seen = []
    servers = [threading.Thread(target=charge_offer_42, args=(use_returning, seen)) for _ in range(8)]
    for s in servers:
        s.start()
    for s in servers:
        s.join()
    label = 'UPDATE ... RETURNING' if use_returning else 'UPDATE, then SELECT '
    print(f'{label}: {len(seen)} charges, {len(set(seen))} distinct balances seen')

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: RETURNING</b></p>
        <p>With <tt>UPDATE</tt>-then-<tt>SELECT</tt>, most readings are someone else's balance: across our runs we saw between 86 and 153 distinct balances for 400 charges. With <tt>RETURNING</tt>, every ad server got its own post-charge balance, 400 of 400, in every run. The single round trip also roughly halved the time per charge.</p>
    </div>
</div>

That is Priya's overspend: each ad server checked a balance, just not its own. A budget check built on `RETURNING` cannot be fooled this way.

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-311</span> &nbsp;<b>Resolved.</b> Budget checks now read the balance their own charge produced, with <tt>UPDATE ... RETURNING</tt>. Priya has her noon answer.
</div>

### What `RETURNING` does not do (yet)

These all fail on engine 10.4:

| You write | You get |
|---|---|
| `RETURNING budget_cents - spend_cents` (an expression) | syntax error 1064 -- return the columns and compute in your app |
| a join, subquery or CTE in the `UPDATE` | 1235 "RETURNING having multiple tables" |
| `ORDER BY ... LIMIT` with `RETURNING` | syntax error 1064 |
| `INSERT ... RETURNING` | syntax error 1064 |
| `RETURNING` inside a stored procedure | compiles, then fails at `CALL` with 1235 |

Column lists, `RETURNING *` and aliases (`RETURNING spend_cents AS s`) work, and inside a transaction a `ROLLBACK` discards the change as usual.

## 3. Expire audience segments with `DELETE ... RETURNING`.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-312</span> &nbsp;<span style="opacity:0.7">opened 9:15 AM</span><br/>
<b>Dana Okafor, Privacy Counsel:</b> When a segment membership expires, we promise advertisers that the person stops being targeted. Last night the database deleted more memberships than the ad server's audit log says it was told about. I need the exact list of what was removed, every time. "Approximately" does not survive an audit.
</div>

The ad server must learn which memberships expired so it stops targeting them. The classic approach, `SELECT` the expired rows and then `DELETE` them, misses any membership that expires between the two statements: it is deleted but never reported. `DELETE ... RETURNING` deletes and reports in one statement, so you get back exactly what was removed.

In [12]:
with connect() as conn, conn.cursor() as cur:
    t0 = time.perf_counter()
    cur.execute('DELETE FROM subscriber_segments WHERE expires_at <= NOW(6) RETURNING subscriber_id, segment')
    removed = cur.fetchall()
    elapsed = time.perf_counter() - t0

print(f'Removed {len(removed):,} expired memberships in {elapsed:.2f} s. First few:')
for subscriber_id, segment in removed[:5]:
    print(f'  subscriber {subscriber_id}: {segment}')

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: the cost of DELETE ... RETURNING</b></p>
        <p>Every removed row comes back to the client, so a large <tt>DELETE ... RETURNING</tt> costs more than a bare <tt>DELETE</tt>. In our runs, removing about 220 thousand rows took 3-8x as long as a bare <tt>DELETE</tt>. For batch jobs that is usually a good trade; for very large purges, return only the key columns you need.</p>
    </div>
</div>

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-312</span> &nbsp;<b>Resolved.</b> The expiry job now sends the ad server exactly the rows it deleted, from the <tt>DELETE</tt> itself. The audit log and the database agree.
</div>

## 4. Give each advertiser's dashboard its own layout with a projection.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-313</span> &nbsp;<span style="opacity:0.7">opened 10:40 AM</span><br/>
<b>Luis Ortega, Sales:</b> Offer 42 is the biggest account we signed during the game. Their marketing team has the dashboard open all day, and every refresh takes long enough to notice. The renewal call is at 3 PM. I would love to not be apologizing on it.
</div>

`notifications` is laid out for the ad server: sharded by `subscriber_id`, sorted by `ts`. A dashboard asking *what did offer 42 do?* finds offer 42's rows on every partition, mixed in with everyone else's, so it reads the whole table.

A **projection** is a second copy of chosen columns, with its own shard key and sort key, kept in sync by the engine on every write. The table keeps the layout its main workload needs, and the projection serves the other one.

We measure rows scanned with `PROFILE`, because at 3 million rows timings are mostly network round trip, while rows scanned grow with your data.

In [13]:
OFFER_WEEK = """
    SELECT DATE(ts) AS day, COUNT(*) AS impressions, SUM(cost_cents) AS spend_cents
    FROM notifications {hint}
    WHERE offer_id = 42 AND ts >= NOW(6) - INTERVAL 7 DAY
    GROUP BY 1 ORDER BY 1"""

OFFER_LATEST = """
    SELECT ts, subscriber_id, cost_cents
    FROM notifications {hint}
    WHERE offer_id = 42 ORDER BY ts DESC LIMIT 20"""


def rows_scanned(sql):
    """Run ``sql`` under PROFILE and return how many rows its table scans read.

    Parameters
    ----------
    sql : str
        A SELECT statement.

    Returns
    -------
    int
        Rows read by ColumnStoreScan operators, summed over all partitions.

    """
    def walk(node):
        if isinstance(node, dict):
            n = 0
            if node.get('executor') == 'ColumnStoreScan' and 'actual_row_count' in node:
                v = node['actual_row_count']
                n += int(v['value'] if isinstance(v, dict) else v)
            return n + sum(walk(v) for v in node.values())
        if isinstance(node, list):
            return sum(walk(v) for v in node)
        return 0

    with connect() as conn, conn.cursor() as cur:
        cur.execute('PROFILE ' + sql)
        cur.fetchall()
        cur.execute('SHOW PROFILE JSON')
        return walk(json.loads(cur.fetchone()[0]))


baseline = {name: rows_scanned(q.format(hint='')) for name, q in
            [('last 7 days', OFFER_WEEK), ('latest 20', OFFER_LATEST)]}
for name, n in baseline.items():
    print(f'Offer 42, {name}: {n:>9,} rows scanned')

Add a projection sharded and sorted by `offer_id`, holding only the four columns the dashboards read:

In [14]:
%%sql
CREATE PROJECTION IF NOT EXISTS notifications_by_offer
    (SHARD KEY (offer_id), SORT KEY (offer_id, ts))
AS SELECT ts, offer_id, subscriber_id, cost_cents FROM notifications;

**Wait for it.** For a few seconds after `CREATE PROJECTION` returns, queries that name the projection fail with error 1877 "projection ... does not exist". Scripts that query a new projection right away should poll first:

In [15]:
def wait_for_projection(table, projection, timeout_s=30):
    """Poll until queries can use ``projection``.

    Parameters
    ----------
    table : str
        The projection's base table.
    projection : str
        The projection name.
    timeout_s : float, optional
        Give up after this many seconds.

    """
    deadline = time.time() + timeout_s
    while True:
        try:
            query(f"SELECT 1 FROM {table} WITH (use_projection='{projection}') LIMIT 1")
            return
        except Exception:
            if time.time() > deadline:
                raise
            time.sleep(0.5)


wait_for_projection('notifications', 'notifications_by_offer')

The optimizer picks a projection by itself only for some query shapes. For a filter on its key, like `offer_id = 42`, name it with a hint after the table (and after its alias, if any): `FROM notifications WITH (use_projection='notifications_by_offer')`.

In [16]:
HINT = "WITH (use_projection='notifications_by_offer')"
for name, q in [('last 7 days', OFFER_WEEK), ('latest 20', OFFER_LATEST)]:
    n = rows_scanned(q.format(hint=HINT))
    print(f'Offer 42, {name}: {baseline[name]:>9,} rows scanned -> {n:>7,} with the projection '
          f'({baseline[name] / n:.0f}x fewer)')

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-313</span> &nbsp;<b>Resolved.</b> Offer 42's dashboard now reads a projection laid out by offer, not the whole table. Luis can take the 3 PM renewal call without apologizing.
</div>

`EXPLAIN` shows why: the query reads one partition (`partitions:single`) instead of all of them, and the scan names the projection:

In [17]:
for line in query('EXPLAIN ' + OFFER_LATEST.format(hint=HINT)):
    print(line[0])

### When the optimizer uses a projection without a hint

- **`GROUP BY` on the projection's shard key** -- picked automatically. Check with `EXPLAIN SELECT offer_id, COUNT(*) FROM notifications GROUP BY offer_id` and look for `PROJECTION notifications_by_offer`.
- **Joins on the projection's shard key** -- picked only when that beats the alternatives. A join to the 356-row `offers` table never used it here, because broadcasting 356 rows is cheaper.
- **Filters and key lookups** -- need the hint, as above.

To rule a projection out for one query, use the hint `WITH (disable_projection_replacement=1)`. To stop the optimizer choosing projections on its own, set `consider_secondary_projection` to `OFF`; hinted queries still use them.

### What a projection costs

Every write to `notifications` now also writes the projection. The next cell copies 300 thousand rows into a fresh table twice, once bare and once with the same projection.

In [18]:
def timed_copy(with_projection):
    """Insert about 300k notifications into a scratch table and return the seconds taken."""
    query('DROP TABLE IF EXISTS notifications_copy')
    query('CREATE TABLE notifications_copy LIKE notifications')
    if with_projection:
        query("""CREATE PROJECTION notifications_copy_by_offer
                 (SHARD KEY (offer_id), SORT KEY (offer_id, ts))
                 AS SELECT ts, offer_id, subscriber_id, cost_cents FROM notifications_copy""")
        wait_for_projection('notifications_copy', 'notifications_copy_by_offer')
    t0 = time.perf_counter()
    query('INSERT INTO notifications_copy SELECT * FROM notifications WHERE subscriber_id MOD 10 = 0')
    elapsed = time.perf_counter() - t0
    if with_projection:
        query('DROP PROJECTION IF EXISTS notifications_copy_by_offer ON notifications_copy')
    query('DROP TABLE notifications_copy')
    return elapsed


print(f'Bulk insert, no projection:   {timed_copy(False):.2f} s')
print(f'Bulk insert, with projection: {timed_copy(True):.2f} s')

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-flask"></b>
    <div>
        <p><b>Lab notes: what a projection costs</b></p>
        <p>In our runs the bulk insert ran 2-10x slower with the projection, and single-row inserts about 20% slower. The factor varies from run to run; that a projection costs something on every write does not. Measure it on your own write path before adding one to a hot table.</p>
    </div>
</div>

Projections pay off when a table is read far more than it is bulk-written, and the reads filter or group on a key the table's layout ignores. They are a Preview feature in this release.

## 5. Make projection DDL re-runnable with `DROP PROJECTION IF EXISTS`.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-314</span> &nbsp;<span style="opacity:0.7">opened 1:40 PM</span><br/>
<b>Sam Whitfield, Data Engineering:</b> The staging refresh I run every afternoon just died on <tt>DROP TABLE notifications</tt>. That script has not changed in months. Did anyone touch that table today?
</div>

Someone did: you, in section 4. Try to drop the table:

In [19]:
try:
    query('DROP TABLE notifications')
except Exception as e:
    print(e)

**A table with a projection cannot be dropped** until the projection is gone. That breaks the usual `DROP TABLE IF EXISTS` / `CREATE TABLE` setup script, and before this release `DROP PROJECTION` had no `IF EXISTS`, so the fix failed on a first run. The re-runnable teardown is now:

In [20]:
%%sql
DROP PROJECTION IF EXISTS notifications_by_offer ON notifications;
DROP PROJECTION IF EXISTS notifications_by_offer ON notifications;

The second statement is a silent no-op. Three details:

- `ON <table>` is required; without it the statement is a syntax error.
- `IF EXISTS` covers a missing projection, not a missing table: if `notifications` does not exist, it fails with 1146. That is why section 1 checks `information_schema.tables` first.
- `CREATE PROJECTION IF NOT EXISTS` is the matching half. `DROP DATABASE` is not blocked by projections.

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-314</span> &nbsp;<b>Resolved.</b> Sam's refresh script now starts with <tt>DROP PROJECTION IF EXISTS ... ON notifications</tt>, and runs cleanly whether or not the projection is there.
</div>

Put the projection back for the rest of the notebook:

In [21]:
query("""CREATE PROJECTION IF NOT EXISTS notifications_by_offer
         (SHARD KEY (offer_id), SORT KEY (offer_id, ts))
         AS SELECT ts, offer_id, subscriber_id, cost_cents FROM notifications""")
wait_for_projection('notifications', 'notifications_by_offer')

## 6. Keep reports and AI agents from slowing ad serving with resource pools.

<div style="border: 1px solid #f0a35e; border-radius: 6px; background: rgba(240,163,94,0.08); padding: 10px 14px; margin: 8px 0;">
<span style="font-family: monospace; opacity: 0.8">ADV-315</span> &nbsp;<span style="opacity:0.7">opened 3:32 PM</span><br/>
<b>Morgan Blaze, Chief Marketing Officer:</b> I demoed the platform to a new investor and the ads LAGGED. On my own phone. Theo in Analytics says it's the AI agent I connected on Friday. The agent, when asked, blamed Theo's campaign report. I don't care whose fault it is. Ads fast, report running, agent running. All three. Today.
</div>

The ad server's query is small: find the highest bid among the offers a subscriber is targeted by. But reports and the agent run on the same workspace, and a heavy join competes with it for the same CPUs.

A **resource pool** is a named set of limits: CPU share, memory, concurrency, a query timeout. Every query runs in one pool, by default `default_pool`, which has no limits. Since the August 2026 release, CPU limits work on workspaces of every size.

Pools belong to the whole workspace, not a database. This section creates two pools and a database user, and section 7 removes them. On a Free Starter Workspace each cell prints a message and skips.

In [22]:
on_starter = bool(shared_tier_check) and shared_tier_check[0][1] == 'ON'
if on_starter:
    print('Free Starter Workspace: resource pools are not available, skipping section 6.')

SERVE = """
    SELECT o.offer_id, o.maximum_bid_cents
    FROM subscriber_segments s JOIN offers o ON CONCAT('buyer:', o.customer) = s.segment
    WHERE s.subscriber_id = %s AND s.expires_at > NOW(6) AND o.enabled
    ORDER BY o.maximum_bid_cents DESC LIMIT 1"""

REPORTS = [
    """SELECT p.vendor, COUNT(DISTINCT r.domain) AS domains, COUNT(*) AS pairs
       FROM requests r JOIN purchases p ON p.subscriber_id = r.subscriber_id
       GROUP BY p.vendor ORDER BY pairs DESC LIMIT 5""",
    """SELECT SUBSTRING_INDEX(domain, '.', -1) AS tld, COUNT(DISTINCT subscriber_id)
       FROM requests WHERE domain RLIKE '^[a-m].*(box|zz|oo).*' GROUP BY 1""",
]

buyers = [r[0] for r in query("SELECT subscriber_id FROM subscriber_segments WHERE segment LIKE 'buyer:%' LIMIT 2000")]

# SingleStore compiles each query shape on first use. Run every query once now,
# so compilation does not land inside the first measurement.
if not on_starter:
    for sql in REPORTS:
        query(sql)
    query(SERVE, (buyers[0],))


def run_reports(pool, stop, done):
    """Run campaign reports in ``pool`` until ``stop`` is set, appending 1 to ``done`` per report."""
    with connect() as conn, conn.cursor() as cur:
        cur.execute(f'SET resource_pool = {pool}')
        i = 0
        while not stop.is_set():
            cur.execute(REPORTS[i % len(REPORTS)])
            cur.fetchall()
            done.append(1)
            i += 1


def serve_ads(report_pool=None, seconds=30, report_threads=4):
    """Serve ads for ``seconds`` while ``report_threads`` reports run in ``report_pool``.

    Parameters
    ----------
    report_pool : str, optional
        Pool the reports run in. With None, no reports run.
    seconds : float, optional
        How long to serve ads.
    report_threads : int, optional
        Number of concurrent report sessions.

    Returns
    -------
    dict
        Serving latency percentiles in milliseconds, and the number of reports completed.

    """
    stop, done = threading.Event(), []
    reporters = [threading.Thread(target=run_reports, args=(report_pool, stop, done))
                 for _ in range(report_threads if report_pool else 0)]
    for r in reporters:
        r.start()
    time.sleep(3 if report_pool else 0)
    latencies = []
    with connect() as conn, conn.cursor() as cur:
        start_reports = len(done)
        end = time.time() + seconds
        while time.time() < end:
            t0 = time.perf_counter()
            cur.execute(SERVE, (random.choice(buyers),))
            cur.fetchall()
            latencies.append((time.perf_counter() - t0) * 1000)
        reports = len(done) - start_reports
    stop.set()
    for r in reporters:
        r.join()
    q = statistics.quantiles(latencies, n=100)
    return {'p50': q[49], 'p95': q[94], 'p99': q[98], 'reports': reports if report_pool else None}


def show(label, r):
    reports = '' if r['reports'] is None else f"   reports completed: {r['reports']}"
    print(f"{label:28s} p50 {r['p50']:5.1f} ms   p95 {r['p95']:5.1f} ms   p99 {r['p99']:5.1f} ms{reports}")

First the baseline: ad serving alone, then with four report sessions in `default_pool` beside it. Each measurement takes about 30 seconds.

In [23]:
if not on_starter:
    results = {'Serving alone': serve_ads()}
    show('Serving alone', results['Serving alone'])
    results['Reports in default_pool'] = serve_ads('default_pool')
    show('Reports in default_pool', results['Reports in default_pool'])

Watch p95 and p99: the slowest ad requests get several times slower while reports run. That is Morgan's lag: most ads were fine, and the slow ones were very slow. Now give the reports their own pool with a soft CPU limit.

**Grant yourself access.** Creating a pool does not let anyone use it, not even you: until you run `GRANT USAGE`, `SET resource_pool` fails with error 1231.

In [24]:
if not on_starter:
    query('CREATE RESOURCE POOL IF NOT EXISTS campaign_reports WITH SOFT_CPU_LIMIT_PERCENTAGE = 20')
    query('GRANT USAGE ON RESOURCE POOL campaign_reports TO CURRENT_USER')
    results['Reports in campaign_reports'] = serve_ads('campaign_reports')
    for label, r in results.items():
        show(label, r)

**Lab notes.** Across six runs, ad-serving latency as a multiple of serving alone:

| | p50 | p95 |
|---|---|---|
| Reports in `default_pool` | 1-5x | 4-7x |
| Reports in a pool with `SOFT_CPU_LIMIT_PERCENTAGE = 20` | about 1x | about 2x |

The pool cut the slow tail by more than half in every run, and the median returned to baseline. **The reports did not slow down**: they completed as many times in their own pool as in `default_pool`. The pool decides who gets the CPU when both want it; it does not throw capacity away.

Before you use pools in production:

- **`SOFT` limits give way when CPUs are idle; `HARD` limits never do.** Hard limits need `SET GLOBAL resource_governor_cpu_limit_mode = 'HARD'`, and then all pools' CPU percentages must add up to 100 or less. That setting affects every pool on the workspace, so this notebook does not change it.
- **The percentage is not a speed limit for a single query.** A report running alone in a 20% pool finished as fast as in `default_pool`; on the smallest workspaces the limit rounds up to a whole CPU per node. It applies when work competes for CPU.
- **`MAX_CONCURRENCY` works per aggregator.** `MAX_CONCURRENCY = 1` still ran two queries at once on a workspace with two aggregators. It is also an effective fence: capping reports at one query each protected serving as well as the CPU limit did.
- **Returning to `default_pool` needs a grant too.** After `SET`ting a session into a custom pool, `SET resource_pool = default_pool` fails with 1231 unless you have `USAGE` on `default_pool`. New sessions start there without one.

### Fence an AI agent with its own login

That leaves Morgan's agent. An AI agent that writes its own SQL is a report you have not read yet: it might forget a join condition, or ask the same question twenty times at once. A pool can hold it to a memory share, a concurrency cap and a query timeout, and a login whose **default** pool is the fenced one means it cannot switch them off.

SingleStore does this for its own agent: if you have used Aura Analyst here, the list below includes an `auraanalyst` pool with a memory cap, a timeout and a queue limit:

In [25]:
if not on_starter:
    print(f"{'pool':24s} {'memory %':>8s} {'timeout s':>9s} {'max conc':>8s} {'soft CPU %':>10s} {'queue':>5s}")
    for row in query("""SELECT pool_name, memory_percentage, query_timeout, max_concurrency,
                               soft_cpu_limit_percentage, max_queue_depth
                        FROM information_schema.resource_pools ORDER BY pool_name"""):
        name, *limits = row
        print(f'{name:24s} ' + ' '.join(f"{'-' if v is None else v:>{w}}" for v, w in zip(limits, [8, 9, 8, 10, 5])))

Create a pool for your agent and a login that lands in it. `WITH` is required before `DEFAULT RESOURCE POOL`, the password must meet the workspace's policy (at least 14 characters on Helios), and the login needs no `USAGE` grant on its own default pool.

In [26]:
import secrets
from urllib.parse import urlparse

if not on_starter:
    # A random token can lack a digit or a symbol, which the workspace's password
    # policy may require, so end it with one character of each kind.
    agent_password = secrets.token_urlsafe(24) + 'Aa1!'
    query("""CREATE RESOURCE POOL IF NOT EXISTS ai_agent
             WITH MEMORY_PERCENTAGE = 20, QUERY_TIMEOUT = 5, MAX_CONCURRENCY = 2, MAX_QUEUE_DEPTH = 4""")
    query('DROP USER IF EXISTS ai_agent')
    query(f"CREATE USER ai_agent IDENTIFIED BY '{agent_password}' WITH DEFAULT RESOURCE POOL = ai_agent")
    query(f'GRANT SELECT ON {DATABASE}.* TO ai_agent')

    # connection_url carries the notebook's own user, which would override user= here,
    # so connect with just its host and port.
    workspace = urlparse(connection_url)


def agent_connect():
    """Connect as the ai_agent login."""
    return s2.connect(host=workspace.hostname, port=workspace.port or 3306,
                      user='ai_agent', password=agent_password, database=DATABASE)

Now act as the agent. Its session starts in the fenced pool and cannot leave, and its bad join, every request matched with every other request by the same subscriber, is stopped by the timeout instead of running for minutes:

In [27]:
RUNAWAY = """SELECT COUNT(*) FROM requests r JOIN requests r2 ON r2.subscriber_id = r.subscriber_id
             WHERE SHA2(CONCAT(r.domain, r2.domain), 256) LIKE 'ab%'"""

if not on_starter:
    with agent_connect() as conn, conn.cursor() as cur:
        cur.execute('SELECT CURRENT_USER(), CURRENT_RESOURCE_POOL()')
        print('Connected as {} in pool {}'.format(*cur.fetchone()))

        for sql in ['SET resource_pool = default_pool', RUNAWAY]:
            t0 = time.perf_counter()
            try:
                cur.execute(sql)
                cur.fetchall()
                print(f'Succeeded: {sql.split()[0]} ...')
            except Exception as e:
                print(f'Refused after {time.perf_counter() - t0:.1f} s: {e}')

The timeout is checked periodically, so the kill (error 2293) lands a few seconds after `QUERY_TIMEOUT`. Killed queries are counted in `information_schema.mv_resource_pool_status`, the place to watch for an agent that keeps hitting its limits:

In [28]:
if not on_starter:
    for pool, killed, finished in query("""SELECT pool_name, SUM(killed_queries), SUM(finished_queries)
                                         FROM information_schema.mv_resource_pool_status
                                         WHERE pool_name IN ('ai_agent', 'campaign_reports') GROUP BY 1"""):
        print(f'{pool:18s} killed {int(killed):3d}   finished {int(finished):4d}')

<div style="border-left: 4px solid #3cb371; background: rgba(60,179,113,0.08); padding: 8px 14px; margin: 8px 0;">
<span style="font-family: monospace">ADV-315</span> &nbsp;<b>Resolved.</b> Theo's reports run in <tt>campaign_reports</tt> with a soft CPU limit, and the agent logs in as <tt>ai_agent</tt>, fenced by memory, concurrency and a timeout it cannot switch off. Ads fast, report running, agent running.
</div>

Zero open tickets. Before you log off, put everything back the way you found it.

## 7. Clean up.

Pools and users belong to the workspace, so dropping the database leaves them behind. Remove the user, then your grant, then the pools: a pool with a `USAGE` grant cannot be dropped, and `REVOKE` fails if the grant is already gone, so each step tolerates having run before.

In [29]:
def best_effort(sql):
    """Run ``sql``, printing instead of raising if it fails."""
    try:
        query(sql)
    except Exception as e:
        print(f'{sql} -> {e}')


if not on_starter:
    best_effort('DROP USER IF EXISTS ai_agent')
    best_effort('REVOKE USAGE ON RESOURCE POOL campaign_reports FROM CURRENT_USER')
    best_effort('DROP RESOURCE POOL IF EXISTS campaign_reports')
    best_effort('DROP RESOURCE POOL IF EXISTS ai_agent')

Drop the database; projections block `DROP TABLE` but not `DROP DATABASE`. On a Free Starter Workspace, remove the tables instead, projection first:

In [30]:
if on_starter:
    for pipeline in ['requests_pipeline', 'purchases_pipeline']:
        query(f'DROP PIPELINE IF EXISTS {pipeline}')
    query('DROP PROJECTION IF EXISTS notifications_by_offer ON notifications')
    for table in ['requests', 'purchases', 'offers', 'notifications', 'subscriber_segments']:
        query(f'DROP TABLE IF EXISTS {table}')
else:
    query('DROP DATABASE IF EXISTS ad_platform')

## 8. Wrap up.

Five tickets, five features, one database:

| Feature | What it is | Use it for | Watch out for |
|---|---|---|---|
| `UPDATE ... RETURNING` | An `UPDATE` that returns the changed rows with their new values | Counters and balances you must read right after changing (ADV-311) | Columns only: no expressions, joins, `ORDER BY ... LIMIT`, or use inside procedures |
| `DELETE ... RETURNING` | A `DELETE` that returns the rows it removed | Expiry and purge jobs that must report exactly what went (ADV-312) | Every row travels back to the client; return only the keys you need |
| Projection | A second copy of chosen columns with its own shard and sort key, kept in sync on every write | A read path that filters or groups on a key the table's layout ignores (ADV-313) | Slower writes; filters need the `use_projection` hint; a few seconds before it is queryable |
| `DROP PROJECTION IF EXISTS ... ON t` | A projection drop that is a no-op when the projection is missing | Setup and teardown scripts that run more than once (ADV-314) | A table with a projection cannot be dropped; `IF EXISTS` does not cover a missing table |
| Resource pool | A named set of limits: CPU share, memory, concurrency, query timeout | Keeping reports from slowing latency-critical queries (ADV-315) | Needs `GRANT USAGE`; `SOFT` limits yield to idle CPU, `HARD` ones never do |
| Login with a `DEFAULT RESOURCE POOL` | A database user whose sessions start in a fenced pool | AI agents and other SQL you did not write (ADV-315) | The pool's timeout fires a few seconds late; watch `mv_resource_pool_status` for kills |

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>